# Customer Churn Prediction
End-to-end scikit-learn pipeline: EDA → preprocessing → modeling → tuning → evaluation → interpretability.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (classification_report, roc_auc_score, precision_recall_curve,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

## 1. Load data
Dataset: [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn). If using Colab, mount Drive or upload the CSV first.

In [2]:
# Uncomment if running in Colab with Drive mounted
# from google.colab import drive
# drive.mount('/content/drive')

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

## 2. Clean data

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df = df.drop('customerID', axis=1)
df.isnull().sum().sum()

## 3. EDA

In [ ]:
print(df['Churn'].value_counts(normalize=True))
sns.countplot(x='Churn', data=df)
plt.title('Churn Distribution')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x='Churn', y='tenure', data=df, ax=axes[0])
sns.boxplot(x='Churn', y='MonthlyCharges', data=df, ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Train/test split

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

## 5. Preprocessing pipeline

In [ ]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [c for c in X.columns if c not in numeric_features]

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

## 6. Baseline model — Logistic Regression

In [ ]:
log_reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))
])

log_reg_pipeline.fit(X_train, y_train)
y_proba = log_reg_pipeline.predict_proba(X_test)[:, 1]
print(classification_report(y_test, log_reg_pipeline.predict(X_test)))
print('ROC-AUC:', roc_auc_score(y_test, y_proba))

## 7. Compare stronger models

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE))
])

hgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(random_state=RANDOM_STATE))
])

for name, pipe in [('Random Forest', rf_pipeline), ('HistGradientBoosting', hgb_pipeline)]:
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    print(f'{name} ROC-AUC: {roc_auc_score(y_test, proba):.3f}')

## 8. Cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(rf_pipeline, X_train, y_train, cv=cv, scoring='roc_auc')
print(f'CV ROC-AUC: {scores.mean():.3f} +/- {scores.std():.3f}')

## 9. Hyperparameter tuning

In [ ]:
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [5, 10, None],
    'classifier__min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(rf_pipeline, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
print('Best CV score:', grid_search.best_score_)

best_model = grid_search.best_estimator_

## 10. Evaluate on test set

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn']).plot()
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
plt.plot(recall, precision)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.show()

## 11. Feature importance

In [ ]:
feature_names = (numeric_features +
    list(best_model.named_steps['preprocessor']
         .named_transformers_['cat']
         .get_feature_names_out(categorical_features)))

result = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring='roc_auc')

importances = pd.Series(result.importances_mean, index=feature_names).sort_values(ascending=False)
importances.head(15).plot(kind='barh', figsize=(8,6))
plt.title('Top 15 Feature Importances (Permutation)')
plt.gca().invert_yaxis()
plt.show()

## 12. Save model

In [ ]:
import joblib
joblib.dump(best_model, '../models/best_model.pkl')